# 01 — Exploratory Data Analysis
Inspect the annotated dataset before training.

In [ ]:
import json, collections
import pandas as pd
import matplotlib.pyplot as plt

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train = load_jsonl('data/annotated/train.jsonl')
val   = load_jsonl('data/annotated/val.jsonl')
test  = load_jsonl('data/annotated/test.jsonl')
print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

In [ ]:
# Label distribution
label_counts = collections.Counter()
for s in train:
    for l in s['labels']:
        if l != 'O': label_counts[l] += 1

df = pd.DataFrame(label_counts.items(), columns=['Label','Count']).sort_values('Count', ascending=False)
df.plot(kind='barh', x='Label', y='Count', figsize=(8,5), title='Bias label distribution (train)')
plt.tight_layout(); plt.show()
print(df.to_string(index=False))

In [ ]:
# Token length distribution
lengths = [len(s['tokens']) for s in train]
pd.Series(lengths).hist(bins=30, figsize=(8,4))
plt.title('Token length distribution'); plt.xlabel('Tokens'); plt.show()
print(f'Mean: {sum(lengths)/len(lengths):.1f} | Max: {max(lengths)} | p95: {sorted(lengths)[int(len(lengths)*0.95)]}')

In [ ]:
# Sample inspection — show a few biased examples
biased = [s for s in train if any(l != 'O' for l in s['labels'])]
for s in biased[:3]:
    flagged = [(t, l) for t, l in zip(s['tokens'], s['labels']) if l != 'O']
    print('TEXT:', s['text'][:120])
    print('FLAGS:', flagged)
    print()